# Tutorial: Complex Geometries

This tutorial covers multi-compartment geometries, importing geometries from existing models, and analytic geometry primitives.

See also the [Complex Geometries](../complex-geometries.md) reference guide.

## Define a multi-compartment model

In [1]:
import pyvcell.vcml as vc

antimony_str = """
    compartment ec = 10000;
    compartment cell = 5000;
    compartment pm = 100;
    compartment nuc = 300;
    compartment nuc_env = 40;
    species A in cell;
    species B in cell;
    J0: A -> B; k1*A - k2*B
    J0 in cell;
    k1 = 0.1; k2 = 0.2
    A = 10
"""

biomodel = vc.load_antimony_str(antimony_str)
model = biomodel.model
model.get_compartment("pm").dim = 2
model.get_compartment("nuc_env").dim = 2
print(model)

2026-03-06T02:42:18.286524Z main WARN The use of package scanning to locate Log4j plugins is deprecated.
Please remove the `packages` attribute from your configuration file.
See https://logging.apache.org/log4j/2.x/faq.html#package-scanning for details.
2026-03-05 21:42:18,290 ERROR (SBMLDocument.java:573) - There was an error accessing the sbml online validator!
2026-03-06T02:42:18.295084Z main WARN The Logger cbit.vcell.model.Kinetics was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@1c9c4ff6 and is now requested with a null message factory (defaults to org.apache.logging.log4j.message.ParameterizedMessageFactory), which may create log events with unexpected formatting.
2026-03-05 21:42:18,297  WARN (SBMLImporter.java:2878) - failed to transform lumped reaction J0 to distributed: linear factor was null, lumped reaction J0 could not be transformed to distributed
2026-03-06T02:42:18.297914Z main WARN The Logger cbit.vcell.mapping.AbstractMathM

## Load geometry from an existing model

In [2]:
tutorial_biomodel = vc.load_vcml_url(
    "https://raw.githubusercontent.com/virtualcell/pyvcell/refs/heads/main/"
    "examples/models/Tutorial_MultiApp_PDE.vcml"
)

tutorial_geometry = tutorial_biomodel.applications[0].geometry

print("Subvolumes:", tutorial_geometry.subvolume_names)
print("Surfaces:", tutorial_geometry.surface_class_names)

Subvolumes: ['ec', 'cytosol', 'Nucleus']
Surfaces: ['cytosol_ec_membrane', 'Nucleus_cytosol_membrane']


## Visualize the imported geometry

In [3]:
tutorial_geometry.plot(save_path="../images/complex-geometry.png")

/Users/jimschaff/Documents/workspace/pyvcell/pyvcell/_internal/geometry/segmented_image_geometry.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Map compartments and simulate

In [4]:
app = biomodel.add_application("app1", geometry=tutorial_geometry)

app.map_compartment("cell", "cytosol")
app.map_compartment("ec", "ec")
app.map_compartment("nuc", "Nucleus")
app.map_compartment("nuc_env", "Nucleus_cytosol_membrane")
app.map_compartment("pm", "cytosol_ec_membrane")

app.map_species("A", init_conc="sin(0.2*x)", diff_coef=1.0)
app.map_species("B", init_conc="cos(0.2*(x+y+z))", diff_coef=1.0)

sim = app.add_sim(name="sim1", duration=2.0, output_time_step=0.5, mesh_size=(50, 50, 50))
results = vc.simulate(biomodel=biomodel, simulation="sim1")

2026-03-06T02:42:19.267081Z main WARN The use of package scanning to locate Log4j plugins is deprecated.
Please remove the `packages` attribute from your configuration file.
See https://logging.apache.org/log4j/2.x/faq.html#package-scanning for details.
2026-03-06T02:42:19.271373Z main WARN The Logger cbit.vcell.model.Kinetics was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@23aef859 and is now requested with a null message factory (defaults to org.apache.logging.log4j.message.ParameterizedMessageFactory), which may create log events with unexpected formatting.


2026-03-06T02:42:19.624880Z main WARN The Logger cbit.vcell.mapping.AbstractMathMapping was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@23aef859 and is now requested with a null message factory (defaults to org.apache.logging.log4j.message.ParameterizedMessageFactory), which may create log events with unexpected formatting.
2026-03-05 21:42:19,627  INFO (DiffEquMathMapping.java:1457) - WARNING:::: MathMapping.refreshMathDescription() ... assigning boundary condition types not unique
2026-03-05 21:42:19,627  INFO (DiffEquMathMapping.java:1457) - WARNING:::: MathMapping.refreshMathDescription() ... assigning boundary condition types not unique
2026-03-05 21:42:19,632  INFO (Entrypoints.java:200) - Returning from vcellToVcml: {"success":true,"message":"Success"}
2026-03-06T02:42:19.648999Z main WARN The use of package scanning to locate Log4j plugins is deprecated.
Please remove the `packages` attribute from your configuration file.
See https:/

Setting Base file name to: `"/Users/jimschaff/Documents/workspace/pyvcell/docs/guides/notebooks/workspace/out_dir_c74l_z9x/SimID_649780665_0_"`
initializing mesh
numVolume=125000

CartesianMesh::computeNormalsFromNeighbors(), compute normals from neighbors
Membrane Elements -> N=8306
qhull precision warning: 
859 has 0 neighbors !
6830 has 0 neighbors !
--------Num of points that have zero neighbors 2
--------Num Neighbors before symmetrize 48504
--------Num Neighbors after symmetrize 53758
Total volume=5921.705897
Total FluxArea =50.44805177
Total FluxAreaXM =0
Total FluxAreaXP =0
Total FluxAreaYM =34.01199592
Total FluxAreaYP =16.43605585
Total FluxAreaZM =0
Total FluxAreaZP =0
mesh initialized
preprocessing finished
pdeCount=2, odeCount=0
No log-file found at constructed path `/Users/jimschaff/Documents/workspace/pyvcell/docs/guides/notebooks/workspace/out_dir_c74l_z9x/SimID_649780665_0_.log`.simulation [SimID_649780665_0_] started
temporary directory used is /var/folders/zz/gcfcvgt

Simulation Complete in Main() ... 


In [5]:
results.plotter.plot_concentrations(save_path="../images/complex-concentrations.png")

/Users/jimschaff/Documents/workspace/pyvcell/pyvcell/sim_results/plotter.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return plt.show()


In [6]:
results.plotter.plot_slice_3d(time_index=0, channel_id="A", save_path="../images/complex-slice3d-A.png")

/Users/jimschaff/Documents/workspace/pyvcell/pyvcell/sim_results/plotter.py:138: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return plt.show()


## Analytic geometry primitives

You can also build geometries from scratch using analytic helpers.

In [7]:
geo = vc.Geometry(name="geo", origin=(0, 0, 0), extent=(10, 10, 10), dim=3)
geo.add_sphere(name="cell_domain", radius=4, center=(5, 5, 5))
geo.add_background(name="ec_domain")
geo.add_surface(name="pm_domain", sub_volume_1="cell_domain", sub_volume_2="ec_domain")

geo.plot(save_path="../images/complex-analytic-geometry.png")

## Clean up

In [8]:
results.cleanup()